# Privacy & Governance Analysis

**Role:** Governance Officer

**Objective:** Assess NovaCred's credit application data for privacy risks, map findings to GDPR and EU AI Act requirements, and propose actionable governance controls.

NovaCred uses machine learning to make credit decisions and recently received a regulatory inquiry about potential discrimination. This notebook evaluates whether the company's data handling practices comply with European data protection law and AI regulation.

We focus on five areas:
1. Identification of personal data (PII) in the dataset
2. Pseudonymization demonstration
3. Mapping to specific GDPR articles
4. EU AI Act classification for credit scoring
5. Concrete governance recommendations

In [1]:
import pandas as pd
import hashlib
import json

# Load the raw dataset directly
with open('../data/raw_credit_applications.json', 'r') as f:
    raw = json.load(f)

df = pd.json_normalize(raw)
print(f"Dataset loaded: {df.shape[0]} records, {df.shape[1]} columns")
df.columns.tolist()

Dataset loaded: 502 records, 21 columns


['_id',
 'spending_behavior',
 'processing_timestamp',
 'applicant_info.full_name',
 'applicant_info.email',
 'applicant_info.ssn',
 'applicant_info.ip_address',
 'applicant_info.gender',
 'applicant_info.date_of_birth',
 'applicant_info.zip_code',
 'financials.annual_income',
 'financials.credit_history_months',
 'financials.debt_to_income',
 'financials.savings_balance',
 'decision.loan_approved',
 'decision.rejection_reason',
 'loan_purpose',
 'decision.interest_rate',
 'decision.approved_amount',
 'financials.annual_salary',
 'notes']

## 1. Personal Data Identification

Under GDPR Article 4, personal data is any information relating to an identified or identifiable natural person. We classify each field by how easily it can identify someone and the risk that creates.

In [2]:
pii_analysis = pd.DataFrame({
    'Field': [
        'full_name', 'email', 'ssn', 'ip_address',
        'date_of_birth', 'zip_code', 'gender', 'spending_behavior'
    ],
    'PII Type': [
        'Direct identifier', 'Direct identifier',
        'Sensitive national identifier', 'Online identifier',
        'Quasi-identifier', 'Quasi-identifier',
        'Special category (Art. 9)', 'Behavioral data'
    ],
    'Risk Level': [
        'High', 'High', 'Critical', 'Medium',
        'Medium', 'Medium', 'High', 'Medium-High'
    ],
    'Why It Matters': [
        'Immediately reveals identity',
        'Unique to an individual, doubles as a contact channel',
        'National ID: unequivocally identifies a person across all systems',
        'Can be linked to a household or individual, especially with timestamps',
        'Combined with zip + gender, can re-identify 87% of US population (Sweeney, 2000)',
        'When paired with DOB/gender, re-identification risk is high',
        'Protected attribute under GDPR Art. 9 and requires explicit consent',
        'Contains categories like Gambling, Alcohol, Adult Entertainment that can reveal addiction, lifestyle, moral choices'
    ]
})

# Display as a clean styled table
pii_analysis.style.set_properties(**{
    'text-align': 'left',
    'font-size': '11px'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left'), ('font-weight', 'bold'), ('background-color', '#f0f0f0')]}
])

,Field,PII Type,Risk Level,Why It Matters
0,full_name,Direct identifier,High,Immediately reveals identity
1,email,Direct identifier,High,"Unique to an individual, doubles as a contact channel"
2,ssn,Sensitive national identifier,Critical,National ID: unequivocally identifies a person across all systems
3,ip_address,Online identifier,Medium,"Can be linked to a household or individual, especially with timestamps"
4,date_of_birth,Quasi-identifier,Medium,"Combined with zip + gender, can re-identify 87% of US population (Sweeney, 2000)"
5,zip_code,Quasi-identifier,Medium,"When paired with DOB/gender, re-identification risk is high"
6,gender,Special category (Art. 9),High,Protected attribute under GDPR Art. 9 and requires explicit consent
7,spending_behavior,Behavioral data,Medium-High,"Contains categories like Gambling, Alcohol, Adult Entertainment that can reveal addiction, lifestyle, moral choices"


### Re-identification Risk

The combination of `zip_code` + `date_of_birth` + `gender` is a well-known re-identification vector. Even after removing direct identifiers like names and SSNs, these three quasi-identifiers alone can uniquely identify most individuals in a population.

Simply "deleting the name column" is not sufficient for privacy protection. NovaCred needs either proper anonymization techniques (k-anonymity, differential privacy) or strict access controls on quasi-identifiers.

### Sensitive Behavioral Data

The `spending_behavior` field contains granular spending categories that raise serious governance concerns. The dataset includes categories such as:

- **Gambling**: can reveal addiction or financial irresponsibility, leading to biased credit decisions
- **Alcohol**: can be used to infer lifestyle or health issues
- **Adult Entertainment**: highly sensitive personal information with no relevance to creditworthiness
- **Healthcare**: can indirectly reveal medical conditions, which is special category data under GDPR Art. 9

These categories have no legitimate role in a credit scoring model. Their presence in the dataset suggests NovaCred is collecting far more behavioral data than necessary, in violation of the data minimization principle (GDPR Art. 5(1)(c)).

In [3]:
# === PII Exposure Audit ===

print("=" * 60)
print("PII EXPOSURE AUDIT")
print("=" * 60)

# 1. SSNs in plain text
ssn_col = 'applicant_info.ssn'
ssn_count = df[ssn_col].notna().sum()
print(f"\n1. SSNs stored in plain text: {ssn_count} / {len(df)} ({ssn_count/len(df)*100:.1f}%)")
print(f"   Sample: {df[ssn_col].dropna().head(3).tolist()}")

# 2. Full names exposed
name_col = 'applicant_info.full_name'
name_count = df[name_col].notna().sum()
print(f"\n2. Full names stored in plain text: {name_count} / {len(df)} ({name_count/len(df)*100:.1f}%)")

# 3. Emails exposed
email_col = 'applicant_info.email'
email_count = df[email_col].notna().sum()
print(f"\n3. Emails stored in plain text: {email_count} / {len(df)} ({email_count/len(df)*100:.1f}%)")

# 4. IP addresses
ip_col = 'applicant_info.ip_address'
ip_count = df[ip_col].notna().sum()
print(f"\n4. IP addresses stored: {ip_count} / {len(df)} ({ip_count/len(df)*100:.1f}%)")

# 5. Sensitive spending categories
sensitive_categories = ['Gambling', 'Alcohol', 'Adult Entertainment', 'Healthcare']
sensitive_count = 0
for behaviors in df['spending_behavior']:
    if isinstance(behaviors, list):
        for b in behaviors:
            if b.get('category') in sensitive_categories:
                sensitive_count += 1
                break
print(f"\n5. Records with sensitive spending categories: {sensitive_count}")
print(f"   Categories flagged: {sensitive_categories}")

# 6. No consent tracking
print(f"\n6. Consent tracking field: NOT PRESENT in dataset")
print(f"   (No field records whether the applicant consented to data processing)")

print("\n" + "=" * 60)

PII EXPOSURE AUDIT

1. SSNs stored in plain text: 497 / 502 (99.0%)
   Sample: ['596-64-4340', '425-69-4784', '370-78-5178']

2. Full names stored in plain text: 502 / 502 (100.0%)

3. Emails stored in plain text: 502 / 502 (100.0%)

4. IP addresses stored: 497 / 502 (99.0%)

5. Records with sensitive spending categories: 91
   Categories flagged: ['Gambling', 'Alcohol', 'Adult Entertainment', 'Healthcare']

6. Consent tracking field: NOT PRESENT in dataset
   (No field records whether the applicant consented to data processing)



## 2. Pseudonymization Demonstration

Pseudonymization replaces identifying values with artificial identifiers so the data can no longer be attributed to a specific person without additional information (GDPR Art. 4(5)).

We use SHA-256 hashing with a salt. The salt prevents rainbow table attacks — pre-computed tables that map common values to their hashes.

In [4]:
def pseudonymize(value, salt="novacred_2025_audit"):
    """SHA-256 hash with salt. Returns first 12 hex characters."""
    if pd.isna(value):
        return None
    return hashlib.sha256(f"{salt}{value}".encode('utf-8')).hexdigest()[:12]

# Apply to the three most critical fields
ssn_col = [c for c in df.columns if 'ssn' in c.lower()][0]
name_col = [c for c in df.columns if 'full_name' in c.lower()][0]
email_col = [c for c in df.columns if 'email' in c.lower()][0]

df['ssn_pseudo'] = df[ssn_col].apply(pseudonymize)
df['name_pseudo'] = df[name_col].apply(pseudonymize)
df['email_pseudo'] = df[email_col].apply(pseudonymize)

# Before vs. after comparison
comparison = df[[ssn_col, 'ssn_pseudo', name_col, 'name_pseudo']].head(5)
comparison.columns = ['Original SSN', 'Pseudonymized SSN', 'Original Name', 'Pseudonymized Name']
comparison

,Original SSN,Pseudonymized SSN,Original Name,Pseudonymized Name
0,596-64-4340,c5fac20022d2,Jerry Smith,f9271ba16b39
1,425-69-4784,a96bd6c86301,Brandon Walker,a4a6a2ccb861
2,370-78-5178,833cb8435189,Scott Moore,762fd86e6ce3
3,194-35-1833,d71d0d3f7e6e,Thomas Lee,bee84b0e3b69
4,480-41-2475,542360cfe634,Brian Rodriguez,6ad8aaa8ba40


### What pseudonymization achieves and doesn't achieve

**Achieves:**
- Prevents casual identification: anyone viewing the dataset cannot read names or SSNs
- Deterministic: the same SSN always maps to the same hash, so duplicate detection and joins still work
- One-way: you cannot reverse the hash to recover the original value

**Does not achieve:**
- **True anonymization.** Under GDPR, pseudonymized data is still personal data because re-identification is possible if the salt is known. Full anonymization would require techniques like k-anonymity or differential privacy.
- **Protection against the data controller.** NovaCred itself could still re-identify records since it holds the original data. Pseudonymization protects against external breaches, not internal misuse.

This distinction matters for compliance because pseudonymized data remains subject to GDPR requirements, while truly anonymous data does not.

### Governance Note on Pseudonymization and Art. 17

A key practical benefit of pseudonymizing PII at the point of collection is that it simplifies compliance with the Right to Erasure (GDPR Art. 17). If an applicant requests deletion:

- **Without pseudonymization (current state):** NovaCred must search for and delete plain-text names, SSNs, and emails across all systems which is operationally complex and error-prone
- **With pseudonymization:** NovaCred only needs to delete the mapping key (the salt or lookup table). The hashed values become meaningless without it, effectively achieving erasure without touching every record

This is a concrete example of how Privacy by Design (Art. 25) reduces compliance burden downstream.

## 3. GDPR Requirements Mapping

We assess NovaCred's data practices against the GDPR provisions most relevant to automated credit scoring.

In [5]:
gdpr_mapping = pd.DataFrame({
    'Article': [
        'Art. 5(1)(c): Data Minimization',
        'Art. 5(1)(e): Storage Limitation',
        'Art. 6(1): Lawful Basis',
        'Art. 9: Special Categories',
        'Art. 17: Right to Erasure',
        'Art. 22: Automated Decisions',
        'Art. 25: Privacy by Design',
        'Art. 35: Impact Assessment'
    ],
    'What It Requires': [
        'Collect only what is adequate, relevant, and necessary',
        'Keep personal data only as long as needed',
        'Have a legal basis: consent, contract, or legitimate interest',
        'Special data (gender, health) needs explicit consent or legal exception',
        'Delete personal data on request when no longer needed',
        'Right not to be subject to purely automated decisions with legal effects',
        'Build data protection into systems from the start',
        'Conduct impact assessment before high-risk processing'
    ],
    'NovaCred Gap': [
        'IP addresses, Gambling/Alcohol/Adult Entertainment spending categories not necessary for credit assessment',
        'No retention policy: 502 records stored indefinitely with no documented expiry',
        'No consent field in the dataset: lawful basis entirely undocumented',
        'Gender collected and influences decisions without Art. 9 exception; spending data can reveal health/addiction',
        '497 SSNs in plain text with no deletion mechanism: erasure requests unfulfillable',
        'All credit decisions automated: rejection_reason is "algorithm_risk_score" with no human review',
        'PII stored unprotected, no encryption or access controls evident',
        'No DPIA conducted despite high-risk automated credit scoring on 502 applicants'
    ],
    'Severity': ['Medium', 'High', 'High', 'High', 'Critical', 'Critical', 'High', 'High']
})

# Display as styled table
gdpr_mapping.style.set_properties(**{
    'text-align': 'left',
    'font-size': '11px'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left'), ('font-weight', 'bold'), ('background-color', '#f0f0f0')]}
])

,Article,What It Requires,NovaCred Gap,Severity
0,Art. 5(1)(c): Data Minimization,"Collect only what is adequate, relevant, and necessary","IP addresses, Gambling/Alcohol/Adult Entertainment spending categories not necessary for credit assessment",Medium
1,Art. 5(1)(e): Storage Limitation,Keep personal data only as long as needed,No retention policy: 502 records stored indefinitely with no documented expiry,High
2,Art. 6(1): Lawful Basis,"Have a legal basis: consent, contract, or legitimate interest",No consent field in the dataset: lawful basis entirely undocumented,High
3,Art. 9: Special Categories,"Special data (gender, health) needs explicit consent or legal exception",Gender collected and influences decisions without Art. 9 exception; spending data can reveal health/addiction,High
4,Art. 17: Right to Erasure,Delete personal data on request when no longer needed,497 SSNs in plain text with no deletion mechanism: erasure requests unfulfillable,Critical
5,Art. 22: Automated Decisions,Right not to be subject to purely automated decisions with legal effects,"All credit decisions automated: rejection_reason is ""algorithm_risk_score"" with no human review",Critical
6,Art. 25: Privacy by Design,Build data protection into systems from the start,"PII stored unprotected, no encryption or access controls evident",High
7,Art. 35: Impact Assessment,Conduct impact assessment before high-risk processing,No DPIA conducted despite high-risk automated credit scoring on 502 applicants,High


### The Two Critical Failures

**Art. 22: Automated Decision-Making:** Credit approval/rejection is a decision that "produces legal effects" on the applicant (they either get a loan or they don't). Under Art. 22, NovaCred must either obtain explicit consent for automated processing, demonstrate contractual necessity, or implement safeguards including the right to human review. The dataset shows no mechanism for any of these.

**Art. 17: Right to Erasure:** If an applicant requests that NovaCred delete their data, the company must be able to locate and remove all their personal information. With SSNs, names, and emails stored in plain text across what may be multiple systems, this is operationally difficult and legally risky. Pseudonymization at the point of collection would significantly reduce this burden.

## 4. EU AI Act Classification

Under Annex III, Section 5(b) of the EU AI Act, AI systems used to evaluate creditworthiness or establish credit scores are classified as <u>high-risk</u>. This applies automatically to any AI system making or influencing credit decisions.

### Compliance obligations for high-risk systems:

| Requirement | AI Act Article | NovaCred Status |
|------------|---------------|-----------------|
| Risk management system | Art. 9 | Not implemented |
| Data quality & governance | Art. 10 | Not adequately addressed |
| Technical documentation | Art. 11 | Not implemented |
| Audit trail / logging | Art. 12 | No decision logging exists |
| Transparency to users | Art. 13 | Applicants not informed of AI use |
| Human oversight mechanism | Art. 14 | No human review process |
| Accuracy & robustness | Art. 15 | No performance metrics available |

NovaCred cannot legally deploy this system in the EU without addressing these gaps. Any evidence of discriminatory outcomes in the credit decisions would compound the compliance failure, as the AI Act explicitly requires that high-risk systems do not produce biased results across protected groups.

## 5. Governance Recommendations

### 5.1 Decision Audit Trail
Log every credit decision with: timestamp, applicant ID, features used, model version, output (approve/deny), and confidence score. This satisfies AI Act Art. 12 and enables responses to individual complaints or regulatory audits.

### 5.2 Human-in-the-Loop
Flag borderline cases for manual review — any application where the model's confidence falls within 10% of the approval threshold. A designated reviewer must have authority to override the model. Required by GDPR Art. 22 and AI Act Art. 14.

### 5.3 Informed Consent & Transparency
Before collecting data, inform applicants that: (a) an AI system will evaluate their application, (b) what data is collected and why, (c) they have the right to request human review. Add a `consent_given` field to the application schema. No such field currently exists.

### 5.4 Data Retention Policy
Define retention periods: application data kept for 7 years (aligned with financial record-keeping norms), then fully anonymized or deleted. Currently no policy exists and data is retained indefinitely.

### 5.5 Minimize Unnecessary Data
Remove or anonymize `ip_address` after initial fraud screening. Aggregate `spending_behavior` into broad categories (essentials, discretionary, financial) rather than storing granular data that can reveal protected characteristics.

### 5.6 Quarterly Bias Monitoring
Compute disparate impact ratios for gender and age at regular intervals. Any DI below 0.8 triggers automatic review and model retraining. This operationalizes the non-discrimination requirement of the AI Act.

## 6. Conclusion

NovaCred's credit scoring system has significant governance gaps across all areas assessed:

**Privacy Exposure:**
- 497 SSNs, 502 full names, and 502 email addresses stored in plain text
- Sensitive spending categories (Gambling, Alcohol, Adult Entertainment) collected without justification
- No consent tracking mechanism exists in the dataset

**GDPR Non-Compliance:**
- Gaps identified across 8 GDPR articles
- Critical failures: automated decisions with no human oversight (Art. 22) and inability to fulfill erasure requests (Art. 17)
- The rejection reason "algorithm_risk_score" confirms purely automated decision-making with legal effects

**EU AI Act:**
- Credit scoring is high-risk under Annex III; all 7 compliance requirements are unmet
- No risk management, no audit trail, no transparency, no human oversight

**Bottom Line:**
NovaCred's system would not survive a regulatory audit in its current form. The recommendations in Section 5 provide a prioritized remediation roadmap, starting with the two most urgent items: implementing human oversight for credit decisions and establishing pseudonymization for all PII fields.